# UBER- Eats Order Tracking Partner Performance Evaluation

In [2]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [3]:
df_orders = pd.read_csv('../Data/016/fct_orders.csv', parse_dates=['expected_delivery_time','actual_delivery_time','order_date'])

pl_orders = pl.read_csv('../Data/016/fct_orders.csv', try_parse_dates=True)

# Pregunta 1

### ¿Cuál es el porcentaje de pedidos entregados a tiempo en enero de 2024? Considera que un pedido está a tiempo si su actual_delivery_time es menor o igual a su expected_delivery_time. Esto nos ayudará a evaluar la precisión general del seguimiento.

In [12]:
df_jan = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
].copy()

df_jan

res = round((df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time']).mean() * 100,2)
# df_jan['succes_delivery'] = df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time']

# success = df_jan[
#     df_jan['succes_delivery'] == True
# ].shape[0]

# res = round((success / df_jan.shape[0]) * 100,2)
 
res

np.float64(66.67)

In [14]:
res = (pl_orders.filter(
    (pl.col('order_date').dt.month() == 1) &
    (pl.col('order_date').dt.year() == 2024)
    )
).select(
    success_delivery = ((pl.col('actual_delivery_time') <= pl.col('expected_delivery_time')).mean() * 100).round(2)
)

# Pregunta 2

### Haz una lista de los 5 mejores repartidores en enero de 2024, clasificados por el mayor porcentaje de entregas a tiempo. Utiliza el campo delivery_partner_name de los registros. Esto nos ayudará a identificar qué socios tienen el mejor desempeño.

```SQL
SELECT
    delivery_partner_name,
    ROUND((COUNT(CASE WHEN (actual_delivery_time <= expected_delivery_time) THEN 1 END) * 100.0 / COUNT(*)),2) AS success_delivery_rate
FROM fct_orders
WHERE ((EXTRACT(MONTH FROM order_date) = 1) AND
       (EXTRACT(YEAR FROM order_date) = 2024))
GROUP BY delivery_partner_name
ORDER BY success_delivery_rate DESC
LIMIT 5;
```

In [21]:
df_jan = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
].copy()

df_jan['is_on_time'] = df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time']

top_5 = (
    df_jan.groupby('delivery_partner_name')['is_on_time']
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

In [27]:
pl_jan = (
    pl_orders.filter(
        (pl.col('order_date').dt.month() == 1) &
        (pl.col('order_date').dt.year() == 2024)
    )
    .group_by('delivery_partner_name')
    .agg(
        success_rate = (
            (pl.col('actual_delivery_time') <= pl.col('expected_delivery_time'))
        .mean() * 100
        ).round(2)
    )
    .sort('success_rate', descending=True)
    .head(5)
)

# Pregunta 3

### Identifica al repartidor (o repartidores) en enero de 2024 cuyo porcentaje de entregas a tiempo sea inferior al 50%. Devuelve sus nombres de repartidor en mayúsculas (UPPERCASE). Necesitamos trabajar con estos socios repartidores para mejorar sus tasas de entrega a tiempo.

```SQL
WITH delivery_success AS (
  SELECT delivery_partner_name,
         ROUND((COUNT(CASE WHEN (actual_delivery_time <= expected_delivery_time) THEN 1 END) *
                100.0 / COUNT(*)), 2) AS success_delivery_rate
  FROM fct_orders
  WHERE ((EXTRACT(MONTH FROM order_date) = 1) AND
         (EXTRACT(YEAR FROM order_date) = 2024))
  GROUP BY delivery_partner_name
)
SELECT
    UPPER(delivery_partner_name)
FROM delivery_success
WHERE success_delivery_rate < 50;
```

In [31]:
df_jan = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
].copy()

df_jan.assign(on_time = df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time'])

df_jan

,order_id,delivery_partner_id,delivery_partner_name,expected_delivery_time,actual_delivery_time,order_date
0,1,1,Alice,2024-01-05 12:00:00,2024-01-05 11:45:00,2024-01-05
1,2,1,Alice,2024-01-12 13:00:00,2024-01-12 13:00:00,2024-01-12
2,3,1,Alice,2024-01-20 12:30:00,2024-01-20 12:35:00,2024-01-20
3,4,2,Bob,2024-01-07 14:00:00,2024-01-07 13:50:00,2024-01-07
4,5,2,Bob,2024-01-15 15:00:00,2024-01-15 15:05:00,2024-01-15
5,6,2,Bob,2024-01-25 14:30:00,2024-01-25 14:00:00,2024-01-25
6,7,3,Charlie,2024-01-03 16:00:00,2024-01-03 15:55:00,2024-01-03
7,8,3,Charlie,2024-01-10 17:00:00,2024-01-10 17:10:00,2024-01-10
8,9,3,Charlie,2024-01-18 16:45:00,2024-01-18 16:40:00,2024-01-18
9,10,4,Dawn,2024-01-04 12:00:00,2024-01-04 12:15:00,2024-01-04
